# 09. Ablation and Robustness

This notebook converts model comparison into a true ablation study. A real ablation isolates each design decision: residual formulation, top-k emphasis, MMD, cell-aware MMD, selection strategy, and representation size. The notebook is designed to aggregate multiple completed runs and summarize component-wise gains, rather than treating baseline-versus-final comparison as an ablation.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 79.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 83.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 33.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 149.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 267.4 kB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [2]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad


In [3]:
import pandas as pd
from pathlib import Path

ABLATION_DIR = RESULTS_DIR / "ablations"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

run_candidates = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
pd.DataFrame({"run_dir": [p.name for p in run_candidates]})


,run_dir
0,residual_base
1,residual_cellaware_mmd


In [4]:
# Expected model variants. Add or rename run directories here once they exist.
expected_runs = {
    "A0_control_copy_baseline": RESULTS_DIR / "canonical" / "baseline_metrics_overall.csv",
    "A1_residual_base": RUNS_DIR / "residual_base" / "final_overall_metrics.csv",
    "A2_residual_cellaware_mmd": RUNS_DIR / "residual_cellaware_mmd" / "final_overall_metrics.csv",
}
available = {k: v for k, v in expected_runs.items() if v.exists()}
pd.DataFrame({"variant": list(available.keys()), "metrics_file": [str(v) for v in available.values()]})


,variant,metrics_file
0,A0_control_copy_baseline,/content/drive/MyDrive/ChemDFM/results/canonic...
1,A1_residual_base,/content/drive/MyDrive/ChemDFM/runs/residual_b...
2,A2_residual_cellaware_mmd,/content/drive/MyDrive/ChemDFM/runs/residual_c...


In [5]:
frames = []
for variant, path in available.items():
    df = pd.read_csv(path).copy()
    df["variant"] = variant
    frames.append(df)

if frames:
    ablation_overall = pd.concat(frames, ignore_index=True)
    ablation_overall.to_csv(ABLATION_DIR / "ablation_overall.csv", index=False)
    display(ablation_overall)
else:
    print("No completed runs found yet. Populate the expected run table after training.")


,split,mode,r2_full,pearson_rowmean,mse,collapse_ratio,mean_shift_error,r2_top20,r2_top50,variant,model
0,test,control_copy,0.600740,0.761599,0.374277,0.635082,0.455507,0.458998,0.542528,A0_control_copy_baseline,NaN
1,test,global_mean,0.602405,0.762905,0.372716,0.633616,0.455401,0.461525,0.544917,A0_control_copy_baseline,NaN
2,test,cell_mean,0.603489,0.763246,0.371700,0.602134,0.455494,0.464252,0.547430,A0_control_copy_baseline,NaN
3,test,drug_lookup,0.617675,0.774740,0.358401,0.648411,0.450282,0.475879,0.557363,A0_control_copy_baseline,NaN
4,ood,control_copy,0.554779,0.734811,0.414899,0.645817,0.471575,0.424265,0.503718,A0_control_copy_baseline,NaN
5,ood,global_mean,0.559701,0.738252,0.410312,0.644630,0.470395,0.430381,0.509394,A0_control_copy_baseline,NaN
6,ood,cell_mean,0.563591,0.739673,0.406687,0.612709,0.469147,0.436299,0.514799,A0_control_copy_baseline,NaN
7,ood,drug_lookup,0.571480,0.747878,0.399335,0.651541,0.466813,0.442601,0.520211,A0_control_copy_baseline,NaN
8,test,NaN,0.636585,0.784522,0.340675,0.640241,0.441033,0.499407,0.577561,A1_residual_base,ResidualDoseResponseModel
9,ood,NaN,0.614290,0.772493,0.359441,0.616443,0.454386,0.484087,0.557015,A1_residual_base,ResidualDoseResponseModel


## Recommended ablation ladder

The final study should include at least the following variants:

- control-copy baseline
- global mean residual baseline
- cell-mean baseline
- residual model with MSE only
- residual model with MSE + top-k
- residual model with global MMD
- residual model with cell-aware MMD
- sensitivity to PCA dimension
- no-control-anchor sanity check

The current notebook provides the aggregation scaffold so that the ablation remains standardized once the runs are produced.


In [6]:
# Seed and hyperparameter robustness scaffold
robustness_plan = pd.DataFrame([
    {"dimension": "seed", "values": "42, 43, 44, 45, 46"},
    {"dimension": "pca_dim", "values": "16, 25, 50, 100"},
    {"dimension": "alpha_mmd", "values": "0.0, 0.01, 0.05, 0.1"},
    {"dimension": "alpha_topk", "values": "0.0, 1.0, 2.0, 4.0"},
])
robustness_plan.to_csv(ABLATION_DIR / "robustness_plan.csv", index=False)
robustness_plan


,dimension,values
0,seed,"42, 43, 44, 45, 46"
1,pca_dim,"16, 25, 50, 100"
2,alpha_mmd,"0.0, 0.01, 0.05, 0.1"
3,alpha_topk,"0.0, 1.0, 2.0, 4.0"
